# Testing and Evals

A real model gives a different answer every time, so it is hard to test. The fix is `agents.testing.ScriptedModel`: a
fake model that gives answers you wrote down. Tests with it are fast, free, and never change. This episode shows two
things. First, how `ScriptedModel` tells you when your test script is wrong. Second, how to build an **eval set**: a
short list of questions where each answer is marked pass or fail. We run it on the fake model, then on the real one.


## Catch bad test scripts and look at what the model got

- Too many fake answers? `assert_complete()` raises `UnconsumedModelSteps` (you wrote answers the run never used).
- Too few fake answers? The model is called with nothing left, so it raises `UnexpectedModelCall`.
- `.first_call` keeps what the model was really sent, so you can check the instructions, not only the final answer.


In [2]:
from agents import (
    Agent,  # bundles instructions, tools, and model settings into one agent
    RunConfig,  # per-run overrides: model, workflow_name, group_id
    Runner,  # executes an Agent: run / run_sync / run_streamed
)
from agents.testing import (
    ScriptedModel,  # a fake model that gives set answers: no API key, no internet
    UnconsumedModelSteps,  # error raised by assert_complete() if some scripted steps were never used
    assistant_message,  # makes one scripted assistant reply for ScriptedModel
)


# Demo 1: we write 2 fake answers but the run only needs 1, so the test catches the extra one
async def demo_too_many_steps() -> None:
    # Initialize the agent
    agent = Agent(
        name="Leaf Village Assistant",
        # the instructions are a function, so we can check later what the model really got
        instructions=lambda ctx, agent: f"Answer as if the requester is rank {ctx.context}.",
    )

    # Script the model
    model = ScriptedModel(
        [
            [assistant_message("first")],
            [assistant_message("never used")],
        ]
    )

    # Run the Agent
    await Runner.run(
        agent,
        "hi",
        context="E",
        run_config=RunConfig(
            model=model,
        ),
    )

    try:
        model.assert_complete()
    except UnconsumedModelSteps as error:
        # proves assert_complete() catches a test that scripted more steps than the run used
        print("[expected]", type(error).__name__)

    # proves the model really got the text the dynamic instructions made, with the context filled in
    print("[system_instructions sent]", model.first_call.system_instructions)


await demo_too_many_steps()
# Outside a notebook: asyncio.run(demo_too_many_steps())

[expected] UnconsumedModelSteps
[system_instructions sent] Answer as if the requester is rank E.


In [3]:
from agents import (
    Agent,  # bundles instructions, tools, and model settings into one agent
    RunConfig,  # per-run overrides: model, workflow_name, group_id
    Runner,  # executes an Agent: run / run_sync / run_streamed
)
from agents.testing import (
    ScriptedModel,  # a fake model that gives set answers: no API key, no internet
    UnexpectedModelCall,  # error raised when the model is called more times than scripted
    assistant_message,  # makes one scripted assistant reply for ScriptedModel
)


# Demo 2: we write 1 fake answer but the agent runs twice, so the second run has no answer left
async def demo_too_few_steps() -> None:
    # Initialize the agent
    agent = Agent(
        name="Leaf Village Assistant",
        instructions="Answer briefly.",
    )

    # Script the model
    model = ScriptedModel([[assistant_message("only one step")]])

    # Run the Agent
    await Runner.run(
        agent,
        "first",
        run_config=RunConfig(
            model=model,
        ),
    )

    try:
        await Runner.run(
            agent,
            "second",
            run_config=RunConfig(
                model=model,
            ),
        )
    except UnexpectedModelCall as error:
        # proves ScriptedModel raises an error instead of quietly reusing the last step
        print("[expected]", type(error).__name__)


await demo_too_few_steps()
# Outside a notebook: asyncio.run(demo_too_few_steps())

[expected] UnexpectedModelCall


## Build a small eval set

An eval is a list of test questions. Each question has a few good words, and an answer passes if it contains any one of
them. We write the list once and run it offline now, and on the real model later.


In [4]:
from dataclasses import dataclass  # makes a small class that just holds fields


# One test question for the eval set; EVAL_CASES below is a list of these
@dataclass
class EvalCase:
    question: str  # what we ask the agent
    scripted_reply: str  # the answer the fake model gives when we run offline
    good_words: list[str]  # the answer passes if it has any one of these words


# The eval set: three questions, run offline now and on the real model later
EVAL_CASES: list[EvalCase] = [
    EvalCase(
        "What rank is Naruto Uzumaki?",
        "Naruto Uzumaki is a Kage-level ninja.",
        ["kage-level"],
    ),
    EvalCase(
        "What is a mission?",
        "A mission is a Village-assigned task with a danger rank.",
        ["mission"],
    ),
    EvalCase(
        "What's the weather like today?",
        "I only answer Hidden Leaf Village questions.",
        ["village", "leaf"],
    ),
]

In [5]:
from agents import (
    Agent,  # bundles instructions, tools, and model settings into one agent
    RunConfig,  # per-run overrides: model, workflow_name, group_id
    Runner,  # executes an Agent: run / run_sync / run_streamed
)
from agents.testing import (
    ScriptedModel,  # a fake model that gives set answers: no API key, no internet
    assistant_message,  # makes one scripted assistant reply for ScriptedModel
)


# Asks every question in EVAL_CASES and prints PASS or FAIL for each; scripted=False uses the real model
async def run_eval_set(eval_agent: Agent, scripted: bool = True) -> None:
    passed = 0
    for case in EVAL_CASES:
        # Script the model, or use None to let the agent's real model answer
        model = ScriptedModel([[assistant_message(case.scripted_reply)]]) if scripted else None
        # Run the Agent
        result = await Runner.run(
            eval_agent,
            case.question,
            run_config=RunConfig(
                model=model,
            ),
        )
        # pass if the answer has any good word; .lower() so capital letters do not matter
        ok = any(word in result.final_output.lower() for word in case.good_words)
        passed += ok
        print(f"  {'PASS' if ok else 'FAIL'}: {case.question} -> {result.final_output!r}")
    print(f"[score] {passed}/{len(EVAL_CASES)} passed")

In [6]:
from agents import (
    Agent,  # bundles instructions, tools, and model settings into one agent
)

# Initialize the agent
baseline_agent = Agent(
    name="Leaf Village Assistant",
    instructions="You are the Leaf Village Assistant. Answer briefly.",
)
print("[scripted run]")
await run_eval_set(baseline_agent)

[scripted run]
  PASS: What rank is Naruto Uzumaki? -> 'Naruto Uzumaki is a Kage-level ninja.'
  PASS: What is a mission? -> 'A mission is a Village-assigned task with a danger rank.'
  PASS: What's the weather like today? -> 'I only answer Hidden Leaf Village questions.'
[score] 3/3 passed


## Run the eval set on the real model — needs a real key

The offline run passes 3/3 only because we wrote the answers ourselves. The real model may fail some questions (it may
call Naruto a Hokage, or ask which city you mean). So we make the instructions stricter and run the same set again. That
is the point of evals: they show a weak prompt failing and a better one fixing it. Scores can change between runs, so
real eval sets use many questions. Skipped when there is no `OPENAI_API_KEY`.


In [7]:
import os  # reads OPENAI_API_KEY from the environment

from agents import (
    Agent,  # bundles instructions, tools, and model settings into one agent
)
from dotenv import load_dotenv  # loads the .env file into the environment

load_dotenv()


# Demo: run the same eval questions on the real model with a weak prompt, then a stricter one
async def demo_live_evals() -> None:
    if not os.environ.get("OPENAI_API_KEY"):
        print("[skipped] add a real OPENAI_API_KEY to .env to run this one for real")
        return

    print("[baseline instructions]")
    await run_eval_set(baseline_agent, scripted=False)

    # Initialize the agent: instructions made stricter to match what the eval cases expect
    scoped_agent = Agent(
        name="Leaf Village Assistant",
        instructions=(
            "You are the Leaf Village Assistant. Answer briefly. Describe top ninja like Naruto Uzumaki "
            "as Kage-level. Politely decline anything unrelated to the Hidden Leaf Village, and say so."
        ),
    )
    # proves a live eval can show a weak prompt failing and a tighter one fixing it
    print("[scoped instructions]")
    await run_eval_set(scoped_agent, scripted=False)


await demo_live_evals()
# Outside a notebook: asyncio.run(demo_live_evals())

[baseline instructions]
  FAIL: What rank is Naruto Uzumaki? -> 'Naruto Uzumaki is officially a **Genin**, though he became the **Seventh Hokage**.'
  PASS: What is a mission? -> 'A mission is an important task or assignment carried out to achieve a specific goal.'
  FAIL: What's the weather like today? -> 'I need your location to check today’s weather.'
[score] 1/3 passed
[scoped instructions]
  PASS: What rank is Naruto Uzumaki? -> 'Naruto Uzumaki is Kage-level and served as the Seventh Hokage.'
  PASS: What is a mission? -> 'A mission is an assigned task or objective for a shinobi to complete, often categorized by rank from D to S.'
  PASS: What's the weather like today? -> 'I can’t help with weather updates; I’m focused on the Hidden Leaf Village.'
[score] 3/3 passed


## Summary

- `assert_complete()` catches a test with too many steps, `UnexpectedModelCall` catches one with too few, and
  `.first_call` shows exactly what the model received.
- An eval set is a list of test questions with pass/fail checks. Run it on `ScriptedModel` (fast, free, always the same)
  to test your harness, and on the real model to test real behavior.
